# 01 — Schema exploration

A first pass at the MIMIC-IV Demo: what's in it, how it's organized, and what we can pull out of it. I use DuckDB to query the gzipped CSVs directly — lightweight on the demo (~100 patients), and the same code will scale to full MIMIC-IV once credentialed.

## Setup

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

DATA_DIR = Path("../data")
HOSP = DATA_DIR / "hosp"
ICU = DATA_DIR / "icu"

con = duckdb.connect()

## Available tables

In [ ]:
hosp_files = sorted([f.name for f in HOSP.glob("*.csv.gz")])
icu_files = sorted([f.name for f in ICU.glob("*.csv.gz")])

print("hosp module:")
for f in hosp_files:
    print(f"  {f}")

print("\nicu module:")
for f in icu_files:
    print(f"  {f}")

MIMIC-IV is split into two modules:

- `hosp/` — hospital-wide tables (patient demographics, admissions, diagnoses, prescriptions, labs).
- `icu/` — ICU-specific tables (ICU stays, charted vitals, inputs/outputs).

For this project I care about: `patients` (demographics), `admissions` (visit-level info incl. outcome), `icustays` (ICU stay boundaries), `chartevents` (vitals over time), `labevents` (labs).

## Patients

In [ ]:
patients = con.execute(f"SELECT * FROM '{HOSP / 'patients.csv.gz'}'").df()
print(f"shape: {patients.shape}")
patients.head()

In [ ]:
patients.dtypes

In [ ]:
patients.isna().mean().sort_values(ascending=False).head(10)

Each row = one patient. Key columns: `subject_id`, `gender`, `anchor_age`, `anchor_year`. MIMIC-IV anchors age for privacy reasons — `anchor_age` is what I'll use to filter adults later.

## Admissions

In [ ]:
admissions = con.execute(f"SELECT * FROM '{HOSP / 'admissions.csv.gz'}'").df()
print(f"shape: {admissions.shape}")
admissions.head()

In [ ]:
# In-hospital mortality is captured by hospital_expire_flag
admissions["hospital_expire_flag"].value_counts()

`hospital_expire_flag` is my outcome: 1 if the patient died in hospital, 0 otherwise. Other useful columns: `admittime`, `dischtime`, `deathtime`, `race`, `insurance`, `marital_status`. The demographic columns are what I'll use to define subgroups in the fairness audit.

## ICU stays

In [ ]:
icustays = con.execute(f"SELECT * FROM '{ICU / 'icustays.csv.gz'}'").df()
print(f"shape: {icustays.shape}")
icustays.head()

In [ ]:
icustays[["los", "first_careunit"]].describe(include="all")

`icustays` gives me ICU stay boundaries. `intime` and `outtime` are what I'll use to extract the "first 24h" features in notebook 04. `los` (length of stay in days) is also worth keeping in mind.

## Quick peek at the time-series tables

In [ ]:
chartevents_preview = con.execute(
    f"SELECT * FROM '{ICU / 'chartevents.csv.gz'}' LIMIT 5"
).df()
chartevents_preview

In [ ]:
labevents_preview = con.execute(
    f"SELECT * FROM '{HOSP / 'labevents.csv.gz'}' LIMIT 5"
).df()
labevents_preview

Both are long-format: one row per (patient, time, measurement). To build features for a 24h window I'll join on `subject_id` + `stay_id` and filter on time relative to `intime` — that's notebook 04.

## Takeaways

- Demo has ~100 patients across `hosp` and `icu`.
- Outcome (`hospital_expire_flag`) lives in `admissions`.
- Demographics are split: `patients` holds `gender` and `anchor_age`; `admissions` holds `race`, `insurance`, `marital_status`.
- Time-series tables are long-format — feature extraction will need a windowed join on `intime`.

Next (notebook 02): demographic distributions, outcome rates, and a first look at how outcomes break down by subgroup.